# 04 — Event Study (Phase 11)

Test `management_qa_divergence` directly against forward abnormal returns,
before any predictive model.

**Honesty check first**: quintile analysis (Q1..Q5) needs at least 5 calls
per bucket to mean anything, and ideally dozens. The Version 0.1 milestone
dataset has only 4 calls. This notebook shows what *can* be computed at
n=4 (the raw scatter, distribution, and a plain correlation), demonstrates
that the quintile code correctly refuses to fabricate 5 groups from 4
points, and validates the quintile/monotonicity/spread logic itself
against a larger synthetic sample (also covered by `tests/test_event_study.py`).
A real quintile event study needs the full universe from Phase 4.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from earnings_nlp.processing.clean_transcripts import parse_transcript_files
from earnings_nlp.features.finbert_features import add_chunk_sentiment, aggregate_call_sentiment
from earnings_nlp.features.change_features import build_divergence_table
from earnings_nlp.backtest.event_returns import load_price_series, build_event_return_table
from earnings_nlp.data.download_transcripts import load_config
from earnings_nlp.utils.paths import TRANSCRIPTS_RAW, DATA_PROCESSED

files = sorted(TRANSCRIPTS_RAW.glob("*.json"))
df = parse_transcript_files(files)
chunk_df = add_chunk_sentiment(df)
call_sentiment = aggregate_call_sentiment(chunk_df)
divergence = build_divergence_table(call_sentiment)

config = load_config()
calls = config["milestone_calls"]
prices_by_ticker = {c["ticker"]: load_price_series(c["ticker"]) for c in calls}
benchmark_prices = load_price_series(config["price_source"]["benchmark_ticker"])
event_returns = build_event_return_table(calls, prices_by_ticker, benchmark_prices)

research_table = divergence.merge(event_returns, on=["ticker", "quarter"])
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
research_table.to_csv(DATA_PROCESSED / "research_table_v0.csv", index=False)

cols = ["ticker", "quarter", "management_qa_divergence", "abnormal_return_1d", "abnormal_return_5d", "abnormal_return_20d"]
research_table[cols]

## What n=4 can actually tell us

A plain correlation and a scatter plot -- nothing more. Four points cannot
distinguish signal from noise.

In [ ]:
for horizon in ["1d", "5d", "20d"]:
    corr = research_table["management_qa_divergence"].corr(research_table[f"abnormal_return_{horizon}"])
    print(f"corr(divergence, abnormal_return_{horizon}) = {corr:.3f}")

In [ ]:
from earnings_nlp.backtest.event_study import plot_divergence_scatter, plot_divergence_distribution
from earnings_nlp.utils.paths import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_divergence_scatter(
    research_table, "management_qa_divergence", "abnormal_return_5d",
    save_path=FIGURES_DIR / "divergence_vs_abnormal_return_5d.png",
)
plot_divergence_distribution(
    research_table, "management_qa_divergence",
    save_path=FIGURES_DIR / "divergence_distribution.png",
)

## Quintiles correctly refuse to run on n=4

This is expected and intentional -- see `assign_divergence_quintiles` in
`src/earnings_nlp/backtest/event_study.py`. Calling 2 groups "quintiles"
would misrepresent how granular the result actually is.

In [ ]:
from earnings_nlp.backtest.event_study import assign_divergence_quintiles

try:
    assign_divergence_quintiles(research_table)
except ValueError as e:
    print("Quintile assignment correctly refused:", e)

## Proving the quintile logic itself works, on synthetic data

Same code path, applied to a larger synthetic sample with a known planted
relationship, so we can check quintile formation, monotonicity, and the
long-short spread all come out correct before trusting them on real data.
This mirrors `tests/test_event_study.py`.

In [ ]:
import numpy as np

from earnings_nlp.backtest.event_study import (
    is_monotonic,
    long_short_spread,
    plot_quintile_mean_returns,
    quintile_return_summary,
)

n = 30
synthetic = pd.DataFrame(
    {
        "management_qa_divergence": np.linspace(-1, 1, n),
        "abnormal_return_5d": -0.02 * np.linspace(-1, 1, n) + np.random.default_rng(0).normal(0, 0.002, n),
    }
)

quintiled = assign_divergence_quintiles(synthetic)
summary = quintile_return_summary(quintiled, ["abnormal_return_5d"])
print(summary)
print()
print("monotonic decreasing:", is_monotonic(summary, "abnormal_return_5d_mean", direction="decreasing"))
print("long-short spread:", long_short_spread(summary, "abnormal_return_5d_mean"))
plot_quintile_mean_returns(summary, "abnormal_return_5d_mean")

## Next step

A genuine Phase 11 result needs the full Phase 4 universe (20 companies x
8 quarters, ~160 calls) downloaded and run through this same pipeline.